# 10. 과거 판례와 HS 지식베이스 통합 검색

09번에서는 과거 완제품 분류 사례의 상품정보와 판정요지를 검색했습니다. 10번에서는 과거 사례와 HS 분류 지식베이스(101개 문서)를 별도 검색 채널로 유지한 뒤, 두 채널의 HS 코드 순위를 RRF(Reciprocal Rank Fusion)로 결합합니다.

신규 제품의 질의에는 제품 속성 5개만 사용합니다. 과거 판례 검색 인덱스는 각 Fold의 학습 사례로만 구성하며, 평가 사례의 판정요지와 정답 HS 코드는 검색에 사용하지 않습니다. 지식베이스는 독립된 규정 자료이므로 모든 Fold에서 동일하게 사용할 수 있습니다.

비교 대상은 ① 판례 검색만 사용, ② 지식베이스 검색만 사용, ③ 두 검색 결과 통합입니다. 이 실험은 후보 코드 추천 성능을 평가하는 것이며 법적 분류 판단을 대체하지 않습니다.


In [ ]:
!pip -q install -U sentence-transformers


In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
kb = pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
target_col = 'HS Code'
summary_col = 'Preprocessed Ruling Summary'
required = [target_col]+input_cols+[summary_col]
missing = [c for c in required if c not in data_all.columns]
if missing: raise ValueError(f'사례 데이터 필수 컬럼 누락: {missing}')
data_all[required] = data_all[required].astype(str).replace('nan','').fillna('')
data_all[target_col] = data_all[target_col].str.strip()
data_all['query_text'] = data_all[input_cols].agg(' '.join,axis=1).str.strip()
data_all['case_product_text'] = data_all['query_text']
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby(target_col).filter(lambda x:len(x)>=4)
group_counts = eligible.groupby(target_col)['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible[target_col].isin(labels)].copy().reset_index(drop=True)
kb_required = ['knowledge_id','hs_code','rule_type','source_sheet','document_text']
missing_kb = [c for c in kb_required if c not in kb.columns]
if missing_kb: raise ValueError(f'지식베이스 필수 컬럼 누락: {missing_kb}')
for c in kb_required: kb[c] = kb[c].fillna('').astype(str)
kb['code_digits'] = kb['hs_code'].map(lambda x:re.sub(r'\\D','',x))
kb['search_text'] = (kb['document_text']+' '+kb['rule_type']+' '+kb['source_sheet']).str.strip()
kb = kb.reset_index(drop=True)
print('평가 사례:',len(data),'| HS 코드:',data[target_col].nunique(),'| 지식 문서:',len(kb),'| 코드 명시 문서:',int(kb['code_digits'].ne('').sum()))
display(data[required].head(3))
display(kb[kb_required].head(3))


## 1. 실험 설계와 코드 후보 생성

판례 채널은 신규 제품의 5개 속성과 학습 Fold의 과거 제품정보를 SBERT로 비교합니다. 검색된 판례의 HS 코드 순위는 사례 순위에 RRF를 적용해 집계합니다.

지식베이스 채널은 동일한 신규 제품 질의와 101개 규정 문서를 비교합니다. HS 코드가 명시된 문서만 코드 후보 순위에 반영합니다. HS 코드가 없는 공통 규정은 코드 후보 점수에 억지로 배정하지 않고 별도 근거 검색 결과로 보존합니다.

각 채널에서 먼저 HS 코드별 순위를 만든 뒤, 채널 간 RRF로 통합합니다. 이렇게 하면 사례 수와 지식 문서 수가 다르더라도 원시 유사도 크기를 직접 더하지 않습니다. RRF의 상수는 60으로 고정합니다.


In [ ]:
MODEL_NAME = 'intfloat/multilingual-e5-base'
RRF_K = 60
TOP_CASES = 20
TOP_KB_DOCS = 30
N_SPLITS = 4
cv = StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)
model = SentenceTransformer(MODEL_NAME)
def code_digits(x): return re.sub(r'\\D','',str(x))
def compatible(a,b): return bool(a and b and (a.startswith(b) or b.startswith(a)))
def rank_codes_from_items(item_order,item_codes,limit=None):
    scores = {}
    selected = item_order if limit is None else item_order[:limit]
    for rank,idx in enumerate(selected,1):
        code = str(item_codes[idx]).strip()
        if code: scores[code] = scores.get(code,0.0)+1/(RRF_K+rank)
    return sorted(scores,key=lambda x:(-scores[x],x)),scores
def fuse_rankings(rankings):
    scores = {}
    for ranking in rankings:
        for rank,code in enumerate(ranking,1): scores[code] = scores.get(code,0.0)+1/(RRF_K+rank)
    return sorted(scores,key=lambda x:(-scores[x],x)),scores
def ranking_metrics(ranking,true_code):
    hits = [i for i,c in enumerate(ranking,1) if compatible(c,true_code)]
    return {'code_mrr':1/hits[0] if hits else 0.0,'code_hit@1':float(bool(hits) and hits[0]==1),'code_hit@3':float(any(i<=3 for i in hits)),'code_hit@5':float(any(i<=5 for i in hits))}
def predict_top(ranking): return ranking[0] if ranking else ''
print('모델:',MODEL_NAME,'| device:',model.device,'| RRF k:',RRF_K)


## 2. Fold별 판례·지식베이스 검색 및 통합

평가 제품의 정답 코드는 검색 함수에 전달하지 않습니다. 정답 코드는 검색 순위가 완성된 뒤 평가 지표를 계산할 때만 사용합니다. 지식베이스 문서는 코드가 명시된 문서와 일반 규정 문서를 구분해 저장합니다.


In [ ]:
result_rows, evidence_rows, fold_rows = [],[],[]
kb_texts = ['passage: '+x for x in kb['search_text'].tolist()]
kb_emb = model.encode(kb_texts,batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[target_col],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr].copy(),data.iloc[te].copy()
    overlap = len(set(train['duplicate_group'])&set(test['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(train),'test_n':len(test),'group_overlap':overlap})
    case_emb = model.encode(['passage: '+x for x in train['case_product_text'].tolist()],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
    query_emb = model.encode(['query: '+x for x in test['query_text'].tolist()],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
    case_sim = query_emb @ case_emb.T
    kb_sim = query_emb @ kb_emb.T
    train_codes = train[target_col].astype(str).to_numpy()
    kb_codes = kb['code_digits'].to_numpy()
    for i in range(len(test)):
        case_order = np.argsort(case_sim[i])[::-1]
        kb_order = np.argsort(kb_sim[i])[::-1]
        case_rank,case_score = rank_codes_from_items(case_order,train_codes,TOP_CASES)
        kb_code_order,kb_score = rank_codes_from_items([j for j in kb_order if kb_codes[j]],kb_codes,TOP_KB_DOCS)
        integrated,integrated_score = fuse_rankings([case_rank,kb_code_order])
        row_id = int(test.index[i])
        true_code = str(test.iloc[i][target_col])
        for method,ranking in [('case_only',case_rank),('kb_only',kb_code_order),('integrated_rrf',integrated)]:
            result_rows.append({'fold':fold,'row_index':row_id,'true_hs':true_code,'method':method,'pred_hs':predict_top(ranking),'candidate_codes':'|'.join(ranking[:10]),**ranking_metrics(ranking,true_code)})
        for rank,idx in enumerate(case_order[:10],1):
            evidence_rows.append({'fold':fold,'row_index':row_id,'channel':'case','rank':rank,'item_id':str(train.index[idx]),'hs_code':train_codes[idx],'similarity':float(case_sim[i,idx]),'evidence_text':train.iloc[idx]['case_product_text'],'ruling_summary':train.iloc[idx][summary_col]})
        for rank,idx in enumerate(kb_order[:15],1):
            evidence_rows.append({'fold':fold,'row_index':row_id,'channel':'knowledge_base','rank':rank,'item_id':kb.iloc[idx]['knowledge_id'],'hs_code':kb.iloc[idx]['hs_code'],'rule_type':kb.iloc[idx]['rule_type'],'source_sheet':kb.iloc[idx]['source_sheet'],'similarity':float(kb_sim[i,idx]),'evidence_text':kb.iloc[idx]['document_text']})
    print('Fold',fold,'완료 | train',len(train),'test',len(test),'| 그룹 중복',overlap)
results = pd.DataFrame(result_rows)
evidence = pd.DataFrame(evidence_rows)
fold_audit = pd.DataFrame(fold_rows)
display(fold_audit)
print('제품별 예측:',len(results),'| 근거 검색 결과:',len(evidence))


## 3. 성능 비교

코드 후보의 정답 일치 여부는 계층적 접두어 호환으로 평가합니다. 이는 검색·후보 순위 평가를 위한 기준이며, 최종 HSK 세번까지의 법적 정답을 의미하지 않습니다. 예측 Top-1의 Accuracy, Macro-F1, Weighted-F1도 함께 확인합니다.


In [ ]:
metric_cols = ['code_mrr','code_hit@1','code_hit@3','code_hit@5']
summary = results.groupby('method')[metric_cols].agg(['mean','std']).round(4)
display(summary)
pred_metrics = []
for method,g in results.groupby('method'):
    pred_metrics.append({'method':method,'accuracy':accuracy_score(g['true_hs'],g['pred_hs']),'macro_f1':f1_score(g['true_hs'],g['pred_hs'],average='macro',zero_division=0),'weighted_f1':f1_score(g['true_hs'],g['pred_hs'],average='weighted',zero_division=0),'empty_prediction_n':int(g['pred_hs'].eq('').sum())})
prediction_metrics = pd.DataFrame(pred_metrics)
display(prediction_metrics.sort_values('accuracy',ascending=False))
summary_flat = results.groupby('method')[metric_cols].agg(['mean','std']).reset_index()
summary_flat.columns = ['_'.join([str(x) for x in col if str(x)]) if isinstance(col,tuple) else col for col in summary_flat.columns]


## 4. 통합 효과 및 근거 유형 점검

아래에서는 통합 결과가 판례 단독 또는 지식베이스 단독보다 나아졌는지 비교합니다. 지식베이스의 일반 규정은 HS 코드가 비어 있으므로 코드 순위에는 포함하지 않았습니다. 다만 실제 서비스에서는 공통 규정의 설명 근거로 중요할 수 있으므로 검색 상위 결과에 별도로 남깁니다.


In [ ]:
pivot = prediction_metrics.set_index('method')
display(pivot[['accuracy','macro_f1','weighted_f1']].round(4))
kb_evidence = evidence[evidence['channel'].eq('knowledge_base')].copy()
kb_evidence['is_code_specific'] = kb_evidence['hs_code'].map(code_digits).ne('')
kb_top_summary = kb_evidence[kb_evidence['rank']<=10].groupby(['rule_type','source_sheet'],dropna=False).size().reset_index(name='top10_count').sort_values('top10_count',ascending=False)
display(kb_top_summary.head(20))
print('상위 10개 지식 문서 중 코드 미기재 비율:',round((~kb_evidence[kb_evidence['rank']<=10]['is_code_specific']).mean(),4))
changed = results.pivot(index=['fold','row_index'],columns='method',values='pred_hs').reset_index()
changed['integrated_changed_vs_case'] = changed['integrated_rrf'].ne(changed['case_only'])
changed['integrated_changed_vs_kb'] = changed['integrated_rrf'].ne(changed['kb_only'])
display(changed[['integrated_changed_vs_case','integrated_changed_vs_kb']].mean().to_frame('rate').round(4))


## 5. 사례별 통합 후보 및 검색 근거 확인

아래 표는 통합 검색 결과와 각 채널의 상위 근거를 확인하기 위한 자료입니다. 지식 문서의 유사도는 의미적 관련성을 나타내며, 해당 규칙이 특정 상품에 법적으로 적용된다는 뜻은 아닙니다. 규칙 유형, 적용 범위, 제외 조건 등을 사람이 검토해야 합니다.


In [ ]:
sample_ids = results[results['method'].eq('integrated_rrf')].head(5)[['fold','row_index']]
display(results[results['row_index'].isin(sample_ids['row_index'])].sort_values(['row_index','method']))
display(evidence[(evidence['channel'].eq('knowledge_base'))&(evidence['rank']<=5)].sort_values(['row_index','rank']).head(40))


## 6. 결과 저장

- `10_integrated_predictions.csv`: 제품별 판례·지식베이스·통합 코드 후보 및 평가 지표
- `10_integrated_evidence.csv`: 사례 및 지식베이스 검색 근거
- `10_integrated_summary.csv`: 코드 후보 순위 지표 요약
- `10_integrated_prediction_metrics.csv`: Top-1 HS 코드 예측 성능
- `10_fold_audit.csv`: Fold별 데이터 분할 및 그룹 중복 점검

판례와 지식베이스의 통합 성능이 단독 채널보다 낮더라도 이를 숨기지 않고 보고합니다. RRF 상수나 검색 문서 수를 조정할 경우 별도 검증 실험으로 분리해야 합니다.


In [ ]:
results.to_csv('10_integrated_predictions.csv',index=False,encoding='utf-8-sig')
evidence.to_csv('10_integrated_evidence.csv',index=False,encoding='utf-8-sig')
summary_flat.to_csv('10_integrated_summary.csv',index=False,encoding='utf-8-sig')
prediction_metrics.to_csv('10_integrated_prediction_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('10_fold_audit.csv',index=False,encoding='utf-8-sig')
print('10번 결과 저장 완료')
